# HealthAI Disease Classifier V3 — Final Fixed
## Bio_ClinicalBERT + Lab Diagnostic Fields

**All issues resolved:**
- **BUG 1 (critical):** `combined[:600]` truncation removed — lab context now reaches the model during training.
- **BUG 2:** Dead `aug_df` replaced with data-quality check.
- **BUG 3:** `CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)` added.
- **BUG 4:** `int()` cast on numpy indices in `predict()`.
- **BUG 5:** Warmup fraction raised to 6% of total steps.
- **NEW FIX:** Missing Step 8 training loop added.
- **NEW FIX:** Hardcoded Supabase credentials removed — now loaded from Colab Secrets / env vars.
- **NEW FIX:** `num_workers` auto-set to 0 on Windows, 2 on Linux/Colab.

**Before running:** Runtime → Change runtime type → T4 GPU

**Credentials setup (Colab):** Runtime → Secrets → add `SUPABASE_URL` and `SUPABASE_KEY` → toggle *Notebook access*


## Step 1 — Install libraries

In [ ]:
!pip install transformers torch scikit-learn supabase matplotlib seaborn nltk --quiet
import nltk
nltk.download('wordnet',quiet=True)
nltk.download('averaged_perceptron_tagger',quiet=True)
nltk.download('punkt',quiet=True)
nltk.download('omw-1.4',quiet=True)
print('All libraries ready.')

All libraries ready.


## Step 2 — Paste credentials

In [2]:
import os

# Store your keys as environment variables (or Colab Secrets) — never hardcode them.
# In Colab: Runtime → Secrets → add SUPABASE_URL and SUPABASE_KEY, then toggle 'Notebook access'
# In local: set them in your shell before launching Jupyter:
#   export SUPABASE_URL='https://xxxx.supabase.co'
#   export SUPABASE_KEY='eyJ...'
try:
    from google.colab import userdata
    SUPABASE_URL = userdata.get('SUPABASE_URL')
    SUPABASE_KEY = userdata.get('SUPABASE_KEY')
    print('Credentials loaded from Colab Secrets.')
except Exception:
    SUPABASE_URL = os.environ.get('SUPABASE_URL', '')
    SUPABASE_KEY = os.environ.get('SUPABASE_KEY', '')
    if SUPABASE_URL and SUPABASE_KEY:
        print('Credentials loaded from environment variables.')
    else:
        raise EnvironmentError(
            'Supabase credentials not found.\n'
            'Set SUPABASE_URL and SUPABASE_KEY as Colab Secrets or environment variables.'
        )


Credentials set.


## Step 3 — Load abstracts + lab data from Supabase

**FIX (BUG 1):** Removed `combined[:600]` character-truncation.
The tokenizer already truncates to `MAX_LEN=512` tokens, so slicing at 600 chars
was silently stripping the `[LAB_CONTEXT]` block out of most training samples.
That meant the model never learnt lab markers during training, but they were
present at inference time → train/inference mismatch → 'always migraine'.

In [3]:
import pandas as pd
from supabase import create_client

# MAX_LEN defined here so data-quality check in Step 4 can reference it
MAX_LEN = 512   # BERT token budget; tokenizer will truncate, not us

client = create_client(SUPABASE_URL, SUPABASE_KEY)
print('Connected to Supabase.')

DISEASE_LABELS = {
    'epilepsy':0,'migraine':1,'stroke':2,'diabetic_neuropathy':3,
    'dementia':4,'parkinsons':5,'dengue':6,'malaria':7,
    'kala_azar':8,'chikungunya':9,'japanese_encephalitis':10,'scrub_typhus':11,
}
LABEL_NAMES = {v:k for k,v in DISEASE_LABELS.items()}
NUM_LABELS  = len(DISEASE_LABELS)

# Load PubMed abstracts
print('Loading abstracts...')
all_rows = []
offset   = 0
while True:
    resp  = client.table('kb_pubmed').select('disease_id,abstract,title').range(offset,offset+199).execute()
    batch = resp.data
    if not batch: break
    all_rows.extend(batch)
    offset += 200
    if len(batch) < 200: break
print(f'Loaded {len(all_rows)} abstracts.')

# Load lab diagnostic data
print('Loading lab diagnostic data...')
lab_rows = {}
try:
    lab_resp = client.table('kb_lab_diagnostics').select('disease_id,key_tests,confirmatory_test,differentiator,raw_lab_text').execute()
    for row in (lab_resp.data or []):
        did = row.get('disease_id','')
        tests = row.get('key_tests',[]) or []
        conf  = row.get('confirmatory_test','') or ''
        diff  = row.get('differentiator','') or ''
        raw   = row.get('raw_lab_text','') or ''
        lab_rows[did] = f"Lab tests: {', '.join(tests[:5])}. Confirmatory: {conf}. {diff}. {raw[:300]}"
    print(f'Loaded lab data for {len(lab_rows)} diseases.')
except Exception as e:
    print(f'Lab data not available ({e}) — using hardcoded fallback.')
    lab_rows = {
        'dengue':      'NS1 antigen positive day 1-5. Platelet count < 100000. IgM positive after day 5. Leucopenia.',
        'migraine':    'NS1 antigen NEGATIVE. Platelet count NORMAL. No fever. Normal CBC. No infection markers.',
        'malaria':     'Blood smear: P. vivax trophozoites or P. falciparum ring forms. RDT pLDH or HRP2 positive.',
        'kala_azar':   'rK39 rapid test positive. Massive splenomegaly. Haemoglobin low. Bone marrow: LD bodies.',
        'scrub_typhus':'Weil-Felix OXK titre > 1:80. IgM ELISA positive. Eschar: black necrotic ulcer. PCR from eschar.',
        'chikungunya': 'IgM ELISA positive after day 5. Polyarthralgia. Lymphopenia. RT-PCR in acute phase.',
        'japanese_encephalitis':'CSF IgM antibody positive. MRI: bilateral thalamic T2 hyperintensities. EEG: diffuse slowing.',
        'epilepsy':    'EEG: spike-wave discharges. MRI brain. Two unprovoked seizures. AED blood levels.',
        'stroke':      'CT brain: infarct or haemorrhage. MRI DWI: acute ischaemic changes. ECG. Blood glucose.',
        'diabetic_neuropathy':'HbA1c > 7%. Nerve conduction study: reduced velocity. 10g monofilament test positive.',
        'dementia':    'MMSE < 24. MoCA low. MRI: hippocampal atrophy. B12 and thyroid normal (reversible causes excluded).',
        'parkinsons':  'DaT scan: reduced uptake. UPDRS score. Bradykinesia + rigidity + tremor. Levodopa response.',
    }
    print('Using hardcoded lab data fallback.')

# ─── BUILD COMBINED TEXT ───────────────────────────────────────────────────
# FIX BUG 1: Do NOT slice combined[:600].  The 600-char cap was cutting the
# [LAB_CONTEXT] block out of nearly every training sample, so the model never
# saw lab markers during training, but they were present at inference time.
# We let the tokenizer truncate to MAX_LEN=512 tokens instead.
records = []
for row in all_rows:
    did      = row.get('disease_id','')
    abstract = (row.get('abstract','') or '').strip()
    if not abstract or did not in DISEASE_LABELS:
        continue
    lab_ctx = lab_rows.get(did, '')
    combined = f"{abstract} [LAB_CONTEXT] {lab_ctx}"   # ← no [:600] truncation
    records.append({'text': combined, 'label': DISEASE_LABELS[did], 'disease': did})

df = pd.DataFrame(records)
print(f'Dataset: {len(df)} samples')
print()
print('Samples per disease:')
print(df['disease'].value_counts().to_string())

Connected to Supabase.
Loading abstracts...
Loaded 2302 abstracts.
Loading lab diagnostic data...
Loaded lab data for 12 diseases.
Dataset: 2302 samples

Samples per disease:
disease
dementia                 200
parkinsons               200
diabetic_neuropathy      199
malaria                  195
kala_azar                193
stroke                   192
migraine                 192
epilepsy                 191
japanese_encephalitis    191
dengue                   188
scrub_typhus             181
chikungunya              180


## Step 4 — Data quality check

**FIX (BUG 2):** Original Step 4 built `aug_df` over ALL rows, then
Step 5 re-augmented only the training split — so `aug_df` was dead code.
Replaced with a quick data-quality check to confirm lab context is present
and text lengths are within the tokenizer's budget.

In [4]:
# Verify lab context is attached and lengths are sane
has_lab = df['text'].str.contains(r'\[LAB_CONTEXT\]', regex=True).sum()
char_mean = df['text'].str.len().mean()
char_max  = df['text'].str.len().max()

# Rough token estimate: 1 word ≈ 1.3 tokens on average
word_mean = df['text'].str.split().str.len().mean()
est_tokens = word_mean * 1.3

print(f'Samples with [LAB_CONTEXT] tag : {has_lab} / {len(df)}')
print(f'Mean text length (chars)       : {char_mean:.0f}')
print(f'Max  text length (chars)       : {char_max}')
print(f'Est. mean token count          : {est_tokens:.0f}  (tokenizer cap = {MAX_LEN})')
if has_lab < len(df):
    print('WARNING: some rows are missing [LAB_CONTEXT] — check lab_rows dict.')
else:
    print('All rows have lab context. ✓')

Samples with [LAB_CONTEXT] tag : 2302 / 2302
Mean text length (chars)       : 1678
Max  text length (chars)       : 4226
Est. mean token count          : 297  (tokenizer cap = 512)
All rows have lab context. ✓


## Step 5 — Train/test split (real data only in test)

In [5]:
import random, re
from nltk.corpus import wordnet
from sklearn.model_selection import train_test_split
random.seed(42)

def get_synonyms(word):
    syns = set()
    for s in wordnet.synsets(word):
        for l in s.lemmas():
            n = l.name().replace('_',' ')
            if n != word and n.isalpha(): syns.add(n)
    return list(syns)

def synonym_replacement(text, n=3):
    # Only replace words before [LAB_CONTEXT] to preserve lab fields
    parts = text.split('[LAB_CONTEXT]')
    if len(parts) < 2: return text
    abstract_part = parts[0]
    lab_part      = '[LAB_CONTEXT]' + parts[1]
    words  = abstract_part.split()
    skip   = {'the','a','an','is','was','were','are','has','had','have','been','with',
               'that','this','from','into','for','and','but','or','not','by','at','on','in','of','to'}
    cands  = [i for i,w in enumerate(words) if len(w)<8 and w.isalpha() and w.lower() not in skip]
    random.shuffle(cands)
    new_words = words.copy()
    replaced  = 0
    for idx in cands:
        if replaced >= n: break
        syns = get_synonyms(words[idx].lower())
        if syns:
            new_words[idx] = random.choice(syns)
            replaced += 1
    return ' '.join(new_words) + ' ' + lab_part

def random_deletion(text, p=0.1):
    parts = text.split('[LAB_CONTEXT]')
    if len(parts) < 2: return text
    words  = parts[0].split()
    if len(words) <= 10: return text
    result = [w for w in words if random.random() > p]
    return (' '.join(result) if result else parts[0]) + ' [LAB_CONTEXT]' + parts[1]

# Split BEFORE augmentation so test set contains only real, unmodified abstracts
train_orig, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Augment training set only (3× expansion)
train_recs = []
for _,row in train_orig.iterrows():
    t,l,d = row['text'],row['label'],row['disease']
    train_recs.append({'text':t,'label':l,'disease':d})
    train_recs.append({'text':synonym_replacement(t,3),'label':l,'disease':d})
    train_recs.append({'text':random_deletion(t,0.1),'label':l,'disease':d})

train_df = pd.DataFrame(train_recs).sample(frac=1,random_state=42).reset_index(drop=True)
print(f'Train (augmented): {len(train_df)}')
print(f'Test  (real only): {len(test_df)}')
print('Note: Test uses ONLY real abstracts with lab context — honest evaluation.')

Train (augmented): 5523
Test  (real only): 461
Note: Test uses ONLY real abstracts with lab context — honest evaluation.


## Step 6 — Load Bio_ClinicalBERT

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = 'emilyalsentzer/Bio_ClinicalBERT'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if str(DEVICE) == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

print(f'Loading {MODEL_NAME}...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForSequenceClassification.from_pretrained(
                MODEL_NAME, num_labels=NUM_LABELS, ignore_mismatched_sizes=True)
model     = model.to(DEVICE)
params    = sum(p.numel() for p in model.parameters())
print(f'Model loaded. Parameters: {params:,}')

Device: cpu
Loading emilyalsentzer/Bio_ClinicalBERT...


config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

c:\Users\BIT\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\BIT\.cache\huggingface\hub\models--emilyalsentzer--Bio_ClinicalBERT. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/436M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: emilyalsentzer/Bio_ClinicalBERT
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Conside

Model loaded. Parameters: 108,319,500


## Step 7 — Tokenize

In [7]:
from torch.utils.data import Dataset, DataLoader

# MAX_LEN=512 is defined in Step 3; repeated here for readability
BATCH_SIZE = 16

class MedDataset(Dataset):
    def __init__(self,texts,labels,tok,ml):
        self.texts,self.labels,self.tok,self.ml=list(texts),list(labels),tok,ml
    def __len__(self): return len(self.texts)
    def __getitem__(self,i):
        e=self.tok(str(self.texts[i]),max_length=self.ml,padding='max_length',
                   truncation=True,return_tensors='pt')
        return {'input_ids':e['input_ids'].squeeze(),
                'attention_mask':e['attention_mask'].squeeze(),
                'label':torch.tensor(self.labels[i],dtype=torch.long)}

train_ds=MedDataset(train_df['text'],train_df['label'],tokenizer,MAX_LEN)
test_ds =MedDataset(test_df['text'], test_df['label'], tokenizer,MAX_LEN)
# num_workers=2 can hang on Windows; use 0 locally, 2 on Linux/Colab
_nw = 0 if platform.system() == 'Windows' else 2
train_dl=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=_nw)
test_dl =DataLoader(test_ds, batch_size=BATCH_SIZE,shuffle=False,num_workers=_nw)
print(f'Train batches: {len(train_dl)}  Test batches: {len(test_dl)}')
print(f'Max tokens: {MAX_LEN} (tokenizer handles truncation — no char-level slice)')

Train batches: 346  Test batches: 29
Max tokens: 512 (tokenizer handles truncation — no char-level slice)


## Step 8 — Fine-tune

**FIX (BUG 3):** Added `CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)`.
- **Class weights** prevent any label from dominating when sample counts differ.
- **Label smoothing (0.05)** prevents the model from becoming over-confident on the
  training set and collapsing to a single class at inference time.
- We call `model(... )` *without* `labels=` so the internal HuggingFace CE loss
  is bypassed, and we apply our weighted criterion instead.

**FIX (BUG 5):** `num_warmup_steps` raised to `int(0.06 * total_steps)` (≈6%).

In [ ]:
import torch.nn as nn
from transformers import get_linear_schedule_with_warmup

EPOCHS    = 4
LR        = 2e-5

# ── Class weights (FIX BUG 3) ──────────────────────────────────────────────
label_counts = train_df['label'].value_counts().sort_index()
class_weights = torch.tensor(
    [1.0 / label_counts[i] for i in range(NUM_LABELS)], dtype=torch.float
).to(DEVICE)
class_weights = class_weights / class_weights.sum() * NUM_LABELS  # normalise
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_dl) * EPOCHS
# FIX BUG 5: warmup = 6% of total steps
warmup_steps = int(0.06 * total_steps)
scheduler    = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

train_losses, train_accs = [], []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for step, batch in enumerate(train_dl):
        optimizer.zero_grad()

        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        labels         = batch['label'].to(DEVICE)

        # Pass WITHOUT labels= so HuggingFace internal CE is bypassed;
        # we apply our weighted criterion below (FIX BUG 3)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss    = criterion(outputs.logits, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs.logits, dim=1)
        correct    += (preds == labels).sum().item()
        total      += labels.size(0)

        if (step + 1) % 50 == 0:
            running_acc = correct / total * 100
            print(f'  Epoch {epoch} | Step {step+1}/{len(train_dl)} '
                  f'| Loss {total_loss/(step+1):.4f} | Acc {running_acc:.1f}%')

    epoch_loss = total_loss / len(train_dl)
    epoch_acc  = correct / total * 100
    train_losses.append(epoch_loss)
    train_accs.append(epoch_acc)
    print(f'Epoch {epoch}/{EPOCHS} — Loss: {epoch_loss:.4f}  Acc: {epoch_acc:.2f}%')

print('Training complete.')


## Step 9 — Evaluate

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import numpy as np, seaborn as sns, matplotlib.pyplot as plt

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_dl:
        out = model(input_ids=batch['input_ids'].to(DEVICE),
                    attention_mask=batch['attention_mask'].to(DEVICE))
        all_preds.extend(torch.argmax(out.logits,dim=1).cpu().numpy())
        all_labels.extend(batch['label'].numpy())

acc = accuracy_score(all_labels, all_preds)
f1m = f1_score(all_labels, all_preds, average='macro')
f1w = f1_score(all_labels, all_preds, average='weighted')
label_list = [LABEL_NAMES[i] for i in range(NUM_LABELS)]

print('='*55)
print('HealthAI V3 — Evaluation Results')
print('='*55)
print(f'Accuracy (overall)  : {acc*100:.2f}%')
print(f'F1 macro            : {f1m:.4f}')
print(f'F1 weighted         : {f1w:.4f}')
print()
print(classification_report(all_labels, all_preds, target_names=label_list))

## Step 10 — Confusion matrix and training curves

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(14,11))
sns.heatmap(cm,annot=True,fmt='d',xticklabels=label_list,yticklabels=label_list,cmap='Blues')
plt.title('HealthAI V3 Confusion Matrix (with lab context)',fontsize=13,fontweight='bold')
plt.ylabel('True'); plt.xlabel('Predicted')
plt.xticks(rotation=45,ha='right'); plt.tight_layout()
plt.savefig('confusion_matrix_v3.png',dpi=150,bbox_inches='tight'); plt.show()

fig,ax=plt.subplots(1,2,figsize=(14,4))
ax[0].plot(range(1,EPOCHS+1),train_losses,'o-',color='firebrick',linewidth=2)
ax[0].set_title('Training Loss',fontweight='bold'); ax[0].set_xlabel('Epoch'); ax[0].grid(alpha=0.3)
ax[1].plot(range(1,EPOCHS+1),train_accs,'o-',color='steelblue',linewidth=2)
ax[1].set_title('Training Accuracy',fontweight='bold'); ax[1].set_xlabel('Epoch'); ax[1].grid(alpha=0.3)
plt.suptitle('HealthAI V3 Training Curves',fontsize=13,fontweight='bold')
plt.tight_layout(); plt.savefig('training_curves_v3.png',dpi=150,bbox_inches='tight'); plt.show()
print('Charts saved.')

## Step 11 — Live demo

**FIX (BUG 4):** `int(i)` cast on numpy array indices before `LABEL_NAMES` lookup,
preventing potential `KeyError` on numpy builds where `np.int64 != int`.

In [ ]:
def predict(text, lab_context=''):
    model.eval()
    combined = f"{text} [LAB_CONTEXT] {lab_context}" if lab_context else text
    enc = tokenizer(combined, max_length=MAX_LEN, padding='max_length',
                    truncation=True, return_tensors='pt')
    with torch.no_grad():
        out = model(input_ids=enc['input_ids'].to(DEVICE),
                    attention_mask=enc['attention_mask'].to(DEVICE))
    probs  = torch.softmax(out.logits, dim=1).cpu().numpy()[0]
    pred   = int(probs.argmax())
    # FIX BUG 4: cast numpy.int64 → Python int for safe dict lookup
    top3   = [(LABEL_NAMES[int(i)], round(float(probs[i])*100, 1))
              for i in probs.argsort()[-3:][::-1]]
    return LABEL_NAMES[pred], round(float(probs[pred])*100, 1), top3

test_cases = [
    ('High fever 39°C, severe headache, retro-orbital pain, rash, myalgia.',
     'NS1 antigen positive. Platelet count 42000. Leucopenia WBC 3200.',
     'dengue'),
    ('Severe unilateral throbbing headache, photophobia, phonophobia, nausea.',
     'NS1 antigen NEGATIVE. Platelet count 210000. No fever. Normal CBC.',
     'migraine'),
    ('High fever 40°C, rigor, sweating, headache, post-monsoon.',
     'Blood smear: P. vivax trophozoites with Schuffner dots. RDT pLDH positive.',
     'malaria'),
    ('High fever 3 weeks, post-monsoon, with eschar on axilla, lymphadenopathy.',
     'Weil-Felix OXK titre 1:160. IgM ELISA scrub typhus positive.',
     'scrub_typhus'),
    ('Prolonged fever 4 weeks, massive splenomegaly, weight loss, Bihar patient.',
     'rK39 rapid test strongly positive. Haemoglobin 7.2 g/dL. Spleen grade IV.',
     'kala_azar'),
    ('Breakthrough seizures. EEG: generalised spike-wave. On levetiracetam.',
     'EEG: 3Hz spike-wave discharges. MRI normal. AED levels subtherapeutic.',
     'epilepsy'),
]

print('HealthAI V3 — Live Predictions (with lab context)')
print('='*65)
correct_demo = 0
for text, lab, expected in test_cases:
    disease, conf, top3 = predict(text, lab)
    tick = 'CORRECT' if disease == expected else 'WRONG'
    if disease == expected: correct_demo += 1
    print(f'Text : {text[:60]}...')
    print(f'Lab  : {lab[:70]}')
    print(f'Pred : {disease.upper()} ({conf}%) [{tick}]  Expected: {expected}')
    print(f'Top3 : {top3}')
    print()

print(f'Demo: {correct_demo}/{len(test_cases)} correct')

## Step 12 — Save model

In [ ]:
import os, zipfile, json as json_lib
from datetime import datetime

SAVE_DIR = 'healthai_classifier_v3'
os.makedirs(SAVE_DIR, exist_ok=True)
model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
with open(f'{SAVE_DIR}/label_map.json','w') as f:
    json_lib.dump(LABEL_NAMES, f, indent=2)
with open(f'{SAVE_DIR}/model_info.json','w') as f:
    json_lib.dump({
        'version':'V3-fixed',
        'base_model':MODEL_NAME,
        'accuracy':round(acc*100,2),
        'f1_macro':round(f1m,4),
        'bugs_fixed':[
            'Removed [:600] char-truncation in Step 3 (primary fix for always-migraine)',
            'Added CrossEntropyLoss with class_weights and label_smoothing=0.05',
            'Fixed numpy.int64 → int cast in predict()',
            'Removed dead aug_df dead code',
            'Raised warmup fraction to 6%',
        ],
        'diseases':list(DISEASE_LABELS.keys()),
        'trained':datetime.now().isoformat()
    }, f, indent=2)

ZIP = 'healthai_model_v3.zip'
with zipfile.ZipFile(ZIP,'w',zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(SAVE_DIR): zf.write(f'{SAVE_DIR}/{fn}', fn)
    for fn in ['confusion_matrix_v3.png','training_curves_v3.png']:
        if os.path.exists(fn): zf.write(fn, fn)

size_mb = os.path.getsize(ZIP)/1e6
print(f'Model saved: {ZIP} ({size_mb:.1f} MB)')
print()
print('To download: Files panel → right-click healthai_model_v3.zip → Download')
print()
print('='*55)
print('V3 TRAINING COMPLETE (fixed)')
print('='*55)
print(f'Accuracy  : {acc*100:.2f}%')
print(f'F1 macro  : {f1m:.4f}')
print(f'Key fix   : Lab context now reaches the model during training')
print('='*55)